# LeetCode #165 — Compare Version Numbers: so sánh từng phần sau khi tách chuỗi

**Mức độ:** Medium · **Chủ đề:** Two Pointers, String

Skeleton trên LeetCode:

```python
class Solution:
    def compareVersion(self, version1: str, version2: str) -> int:
```

Mục tiêu của notebook:

1. Hiểu cách so sánh phiên bản: so **từng phần** (revision) như **số nguyên**, từ trái sang phải.
2. Hai bẫy: **số 0 ở đầu** (`"01" = "1"`) và **phần thiếu coi như 0** (`"1.0" = "1"`).
3. Cách 1: `split('.')` + `int` + bù 0. Cách 2: **hai con trỏ** đọc số trực tiếp, `O(1)` bộ nhớ.
4. Mở rộng: sắp xếp danh sách phiên bản, so sánh số rất dài không đổi sang `int`.

# 1. Bài toán là gì?

Cho hai chuỗi phiên bản `version1` và `version2`. Mỗi chuỗi gồm các **phần** (revision) ngăn cách bởi dấu chấm `'.'`; giá trị của một phần là **số nguyên** của nó, bỏ qua số 0 ở đầu.

So sánh: đi từ trái sang, so lần lượt các phần cùng vị trí. Nếu một chuỗi có ít phần hơn, các phần thiếu coi là `0`.

Trả về `-1` nếu `version1 < version2`, `1` nếu `version1 > version2`, ngược lại `0`.

**Constraints (theo đề):**

- `1 <= version1.length, version2.length <= 500`
- Chỉ gồm chữ số và `'.'`.
- Là chuỗi phiên bản hợp lệ.
- Mọi phần đều vừa số nguyên 32 bit.

---

## Ví dụ

```text
"1.2",     "1.10"     ->  -1    (2 < 10)
"1.01",    "1.001"    ->   0    (01 và 001 đều là 1)
"1.0",     "1.0.0.0"  ->   0    (phần thiếu là 0)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Chuỗi (string) trong Python
2. Hai con trỏ (two pointers)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Chuỗi (string) trong Python

### Chuỗi là bất biến

`str` **không sửa được** tại chỗ: `s[0] = "x"` gây lỗi. Mọi "sửa" đều tạo chuỗi mới. Vì vậy:

- Ghép nhiều mảnh: gom vào **list** rồi `"".join(list)` (`O(n)`), tránh `s += ...` trong vòng lặp (có thể `O(n²)`).
- Cần sửa từng ký tự: `chars = list(s)`, sửa, rồi `"".join(chars)`.

In [1]:
s_str = "hello"
try:
    s_str[0] = "H"
except TypeError as e:
    print("Sửa trực tiếp:", e)
chars_str = list(s_str); chars_str[0] = "H"
print("qua list:", "".join(chars_str))
manh = []
for i in range(5):
    manh.append(str(i))
print("join:", "-".join(manh))

Sửa trực tiếp: 'str' object does not support item assignment
qua list: Hello
join: 0-1-2-3-4


### Cắt chuỗi (slicing) và các thao tác hay dùng

| Biểu thức | Ý nghĩa |
|---|---|
| `s[i:j]` | ký tự từ `i` tới `j−1` (tạo chuỗi mới, `O(j−i)`) |
| `s[::-1]` | chuỗi đảo ngược |
| `s.split()` | tách theo khoảng trắng (bỏ khoảng trắng thừa) |
| `s.split(",")` | tách theo dấu phẩy (giữ phần rỗng) |
| `s.strip()` | bỏ khoảng trắng hai đầu |
| `s.startswith(p)`, `s.find(p)` | kiểm tra tiền tố, tìm vị trí |
| `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()` | phân loại ký tự |
| `c.lower()`, `c.upper()` | đổi hoa/thường |

In [2]:
t_str = "  Xin chào   thế giới  "
print(repr(t_str.split()), "|", repr(t_str.strip()), "|", repr("a,,b".split(",")))
print("đảo:", "abcdef"[::-1], "| cắt [1:4]:", "abcdef"[1:4], "| 'abc'.find('c') =", "abc".find("c"))
print([(c, c.isdigit(), c.isalpha()) for c in "a1 "])

['Xin', 'chào', 'thế', 'giới'] | 'Xin chào   thế giới' | ['a', '', 'b']
đảo: fedcba | cắt [1:4]: bcd | 'abc'.find('c') = 2
[('a', False, True), ('1', True, False), (' ', False, False)]


### Ký tự và mã số: `ord` / `chr`

Mỗi ký tự có một mã số (Unicode). `ord("a") = 97`, `chr(97) = "a"`. Mẹo: chữ số `c` → giá trị `ord(c) - ord("0")`; chữ thường `c` → chỉ số 0..25 là `ord(c) - ord("a")`.

In [3]:
print("ord('a') =", ord("a"), "| chr(98) =", chr(98))
print("'7' → 7:", ord("7") - ord("0"), "| 'd' → chỉ số:", ord("d") - ord("a"))
dem26 = [0] * 26
for c in "banana":
    dem26[ord(c) - ord("a")] += 1
print("đếm 26 chữ cái của 'banana' (a, b, n):", dem26[0], dem26[1], dem26[13])

ord('a') = 97 | chr(98) = b
'7' → 7: 7 | 'd' → chỉ số: 3
đếm 26 chữ cái của 'banana' (a, b, n): 3 1 2


### Gom nhóm ký tự liên tiếp

`itertools.groupby` gom các phần tử **liên tiếp bằng nhau** — tiện cho mã hóa độ dài loạt (run-length): `"aaabcc" → 3a 1b 2c`. Tự viết bằng hai con trỏ: `i` ở đầu nhóm, `j` chạy tới khi ký tự khác.

In [4]:
from itertools import groupby
print([(k, len(list(g))) for k, g in groupby("aaabcc")])

def run_length(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        out.append(f"{j - i}{s[i]}")
        i = j
    return "".join(out)
print("run_length('aaabcc') =", run_length("aaabcc"))

[('a', 3), ('b', 1), ('c', 2)]
run_length('aaabcc') = 3a1b2c


Lỗi hay gặp: `split(" ")` sinh phần rỗng khi có nhiều dấu cách; so sánh chuỗi số `"10" < "9"` là **True** (so từ điển, không phải số); nối chuỗi trong vòng lặp lớn; quên chuỗi rỗng.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [5]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [6]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Áp dụng vào bài này

Mỗi phiên bản là một chuỗi các số nối bằng dấu chấm: dùng `split(".")` và `int(...)` (tự bỏ số 0 đứng đầu) để có danh sách số, rồi so **từng vị trí** như so thứ tự từ điển, coi phần thiếu là `0`. Cách tối ưu bộ nhớ dùng **hai con trỏ** đọc song song hai chuỗi, mỗi lần đọc một "phần" bằng cách ghép chữ số.

# 3. Nền tảng

In [7]:
from typing import List
import random, functools

print('"1.10".split(".") =', "1.10".split("."))
print('int("001") =', int("001"), '| int("0") =', int("0"))
print('so sánh chuỗi "2" < "10"?', "2" < "10", "← sai về mặt số! (so từ điển: '2' > '1')")
print('so sánh số 2 < 10?', 2 < 10)

"1.10".split(".") = ['1', '10']
int("001") = 1 | int("0") = 0
so sánh chuỗi "2" < "10"? False ← sai về mặt số! (so từ điển: '2' > '1')
so sánh số 2 < 10? True


Bẫy lớn nhất: so **chuỗi** `"2"` với `"10"` theo thứ tự từ điển cho `"2" > "10"`. Phải đổi sang **số** trước khi so.

# 4. Cách 1: tách, đổi sang số, bù 0

```text
a = [int(x) for x in v1.split(".")]
b = [int(x) for x in v2.split(".")]
bù 0 cho list ngắn hơn
so từng cặp; khác nhau thì trả -1/1; hết mà bằng → 0
```

In [8]:
def compare_split(v1: str, v2: str) -> int:
    a = [int(x) for x in v1.split(".")]
    b = [int(x) for x in v2.split(".")]
    n = max(len(a), len(b))
    a += [0] * (n - len(a))
    b += [0] * (n - len(b))
    for x, y in zip(a, b):
        if x != y:
            return -1 if x < y else 1
    return 0

for v1, v2 in [("1.2", "1.10"), ("1.01", "1.001"), ("1.0", "1.0.0.0"), ("0.1", "1.1"), ("1.0.1", "1")]:
    print(f"{v1!r:>9} vs {v2!r:<10}: {compare_split(v1, v2):>2}")

    '1.2' vs '1.10'    : -1
   '1.01' vs '1.001'   :  0
    '1.0' vs '1.0.0.0' :  0
    '0.1' vs '1.1'     : -1
  '1.0.1' vs '1'       :  1


`itertools.zip_longest(a, b, fillvalue=0)` làm việc bù 0 gọn hơn. Python còn so **list** theo thứ tự từ điển, nhưng cần bỏ các số 0 ở **cuối** trước (`[1, 0] > [1]` theo cách so list).

In [9]:
from itertools import zip_longest
print(list(zip_longest([1, 2], [1, 2, 0, 3], fillvalue=0)))
print("[1, 0] > [1]?", [1, 0] > [1], "← list dài hơn được coi là lớn hơn, nên phải bỏ số 0 cuối")

[(1, 1), (2, 2), (0, 0), (0, 3)]
[1, 0] > [1]? True ← list dài hơn được coi là lớn hơn, nên phải bỏ số 0 cuối


# 5. Cách 2: hai con trỏ, đọc số trực tiếp

Không cần tạo list: mỗi chuỗi có một con trỏ; mỗi vòng **đọc một phần** (ghép chữ số `x = x*10 + d` tới khi gặp `'.'` hoặc hết chuỗi), rồi bỏ qua dấu chấm. Chuỗi đã hết thì phần đọc được là `0`.

```text
i = j = 0
while i < len(v1) or j < len(v2):
    x = đọc số từ v1 tại i (0 nếu hết)
    y = đọc số từ v2 tại j (0 nếu hết)
    if x != y: return -1 if x < y else 1
return 0
```

In [10]:
def read_revision(s: str, i: int):
    """Đọc một phần bắt đầu tại i; trả về (giá trị, vị trí sau dấu chấm)."""
    x = 0
    while i < len(s) and s[i] != ".":
        x = x * 10 + (ord(s[i]) - ord("0"))
        i += 1
    return x, i + 1                       # bỏ qua dấu chấm (hoặc vượt cuối)


def compare_two_pointers(v1: str, v2: str) -> int:
    i = j = 0
    while i < len(v1) or j < len(v2):
        x, i = read_revision(v1, i) if i < len(v1) else (0, i)
        y, j = read_revision(v2, j) if j < len(v2) else (0, j)
        if x != y:
            return -1 if x < y else 1
    return 0

for v1, v2 in [("1.2", "1.10"), ("1.01", "1.001"), ("1.0", "1.0.0.0"), ("0.1", "1.1"), ("1.0.1", "1")]:
    print(f"{v1!r:>9} vs {v2!r:<10}: {compare_two_pointers(v1, v2):>2}")

    '1.2' vs '1.10'    : -1
   '1.01' vs '1.001'   :  0
    '1.0' vs '1.0.0.0' :  0
    '0.1' vs '1.1'     : -1
  '1.0.1' vs '1'       :  1


# 6. Mô phỏng

In [11]:
def trace_compare(v1, v2):
    i = j = 0
    k = 0
    while i < len(v1) or j < len(v2):
        x, i = read_revision(v1, i) if i < len(v1) else (0, i)
        y, j = read_revision(v2, j) if j < len(v2) else (0, j)
        k += 1
        verdict = "bằng, đi tiếp" if x == y else ("v1 nhỏ hơn → -1" if x < y else "v1 lớn hơn → 1")
        print(f"   phần {k}: {x} vs {y} → {verdict}")
        if x != y:
            return -1 if x < y else 1
    print("   hết các phần, tất cả bằng → 0")
    return 0

for pair in [("1.01", "1.001"), ("1.0", "1.0.0.0"), ("7.5.2.4", "7.5.3")]:
    print(pair)
    trace_compare(*pair)

('1.01', '1.001')
   phần 1: 1 vs 1 → bằng, đi tiếp
   phần 2: 1 vs 1 → bằng, đi tiếp
   hết các phần, tất cả bằng → 0
('1.0', '1.0.0.0')
   phần 1: 1 vs 1 → bằng, đi tiếp
   phần 2: 0 vs 0 → bằng, đi tiếp
   phần 3: 0 vs 0 → bằng, đi tiếp
   phần 4: 0 vs 0 → bằng, đi tiếp
   hết các phần, tất cả bằng → 0
('7.5.2.4', '7.5.3')
   phần 1: 7 vs 7 → bằng, đi tiếp
   phần 2: 5 vs 5 → bằng, đi tiếp
   phần 3: 2 vs 3 → v1 nhỏ hơn → -1


# 7. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def compareVersion(self, version1: str, version2: str) -> int:
        a = [int(x) for x in version1.split(".")]
        b = [int(x) for x in version2.split(".")]
        for x, y in zip_longest(a, b, fillvalue=0):
            if x != y:
                return -1 if x < y else 1
        return 0

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.compareVersion("1.2", "1.10"), -1)
check("Ví dụ 2", sol.compareVersion("1.01", "1.001"), 0)
check("Ví dụ 3", sol.compareVersion("1.0", "1.0.0.0"), 0)
check("Lớn hơn", sol.compareVersion("1.0.1", "1"), 1)
check("Phần đầu", sol.compareVersion("0.1", "1.1"), -1)

def random_version():
    return ".".join(str(random.randint(0, 12)).zfill(random.randint(1, 3)) for _ in range(random.randint(1, 4)))

def reference(v1, v2):
    a = [int(x) for x in v1.split(".")]
    b = [int(x) for x in v2.split(".")]
    while a and a[-1] == 0: a.pop()
    while b and b[-1] == 0: b.pop()
    return (a > b) - (a < b)

random.seed(165)
for _ in range(10000):
    v1, v2 = random_version(), random_version()
    exp = reference(v1, v2)
    assert sol.compareVersion(v1, v2) == exp == compare_two_pointers(v1, v2) == compare_split(v1, v2), (v1, v2)
print("10000 cặp ngẫu nhiên: ba cách khớp nhau")

Ví dụ 1          PASS  -1
Ví dụ 2          PASS  0
Ví dụ 3          PASS  0
Lớn hơn          PASS  1
Phần đầu         PASS  -1


10000 cặp ngẫu nhiên: ba cách khớp nhau


# 8. Vì sao đúng? Độ phức tạp?

Theo định nghĩa của đề, phiên bản tương đương dãy số nguyên vô hạn `(p1, p2, …, pk, 0, 0, …)`; so sánh là so **từ điển** hai dãy này. Bù 0 cho bằng độ dài rồi so từng cặp đúng định nghĩa đó; phần tử khác nhau **đầu tiên** quyết định kết quả.

```text
Cách 1:  Time O(n + m), Space O(n + m) (list các phần)
Cách 2:  Time O(n + m), Space O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: So các phần dưới dạng **chuỗi**

In [14]:
def wrong_string_compare(v1, v2):
    a, b = v1.split("."), v2.split(".")
    for x, y in zip_longest(a, b, fillvalue="0"):
        if x != y:
            return -1 if x < y else 1                # SAI: so từ điển
    return 0

for v1, v2 in [("1.2", "1.10"), ("1.01", "1.001")]:
    print(f"{v1!r} vs {v2!r}: sai {wrong_string_compare(v1, v2):>2} | đúng {sol.compareVersion(v1, v2):>2}")

'1.2' vs '1.10': sai  1 | đúng -1
'1.01' vs '1.001': sai  1 | đúng  0


`"2" > "10"` (từ điển) và `"01" != "001"` (chuỗi khác nhau) — cả hai đều sai.

## Lỗi 2: Không bù 0, chỉ so phần chung rồi so độ dài

In [15]:
def wrong_length(v1, v2):
    a = [int(x) for x in v1.split(".")]
    b = [int(x) for x in v2.split(".")]
    for x, y in zip(a, b):
        if x != y:
            return -1 if x < y else 1
    return (len(a) > len(b)) - (len(a) < len(b))     # SAI: "1.0" và "1" có độ dài khác nhau

print("Sai :", wrong_length("1.0", "1.0.0.0"), "| Đúng:", sol.compareVersion("1.0", "1.0.0.0"))

Sai : -1 | Đúng: 0


## Lỗi 3: So cả list bằng toán tử `<` của Python mà không bỏ số 0 cuối

In [16]:
def wrong_list_compare(v1, v2):
    a = [int(x) for x in v1.split(".")]
    b = [int(x) for x in v2.split(".")]
    return (a > b) - (a < b)                          # SAI: [1, 0] > [1]

print("Sai :", wrong_list_compare("1.0", "1"), "| Đúng:", sol.compareVersion("1.0", "1"))

Sai : 1 | Đúng: 0


## Lỗi 4: Bỏ toàn bộ dấu chấm rồi so số

`"1.10"` → `110`, `"11.0"` → `110` — hai phiên bản khác nhau thành cùng một số. Dấu chấm mang thông tin ranh giới, không bỏ được.

# 10. Mẹo nhận diện pattern

- **So sánh theo từng thành phần** (phiên bản, ngày tháng, địa chỉ IP) → tách thành dãy, so **từ điển** các thành phần **đã chuẩn hóa** (đổi sang số, bù mặc định).
- **Không muốn tạo bản sao** → hai con trỏ đọc từng thành phần.
- **Cần sắp xếp theo tiêu chí tự định nghĩa** → `key=` trả về tuple chuẩn hóa, hoặc `functools.cmp_to_key`.

Bài liên quan: **#468 Validate IP Address**, **#179 Largest Number** (so sánh tự định nghĩa), **#1065**, **#929 Unique Email Addresses** (chuẩn hóa chuỗi).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

So sánh `"7.5.2.4"` với `"7.5.3"`, và `"1.0.0.1"` với `"1"`.

## Bài 2 — Điền chỗ trống

In [17]:
def exercise_2_compare_version(version1: str, version2: str) -> int:
    a = [int(x) for x in version1.split(".")]
    b = [int(x) for x in version2.split(".")]
    # TODO: so từng cặp (bù 0); trả -1/1 khi khác; cuối cùng trả 0
    pass


# assert exercise_2_compare_version("1.2", "1.10") == -1
# assert exercise_2_compare_version("1.01", "1.001") == 0
# assert exercise_2_compare_version("1.0", "1.0.0.0") == 0
# assert exercise_2_compare_version("1.0.1", "1") == 1

## Bài 3 — Biến thể: sắp xếp danh sách phiên bản

Sắp xếp tăng dần; các phiên bản **bằng nhau** giữ thứ tự ban đầu (sắp xếp ổn định).

In [18]:
def exercise_3_sort_versions(versions: List[str]) -> List[str]:
    # TODO
    pass


# assert exercise_3_sort_versions(["1.10", "1.2", "1.0.5", "1"]) == ["1", "1.0.5", "1.2", "1.10"]
# assert exercise_3_sort_versions(["1.0", "1", "0.9"]) == ["0.9", "1.0", "1"]

## Bài 4 — Biến thể: đếm số phần **thực sự có nghĩa**

Trả về số phần của phiên bản sau khi bỏ các phần bằng 0 ở **cuối** (`"1.2.0.0"` → 2, `"0.0"` → 0). Hai phiên bản bằng nhau khi dãy đã chuẩn hóa này bằng nhau.

In [19]:
def exercise_4_significant_parts(version: str) -> int:
    # TODO
    pass


# assert exercise_4_significant_parts("1.2.0.0") == 2
# assert exercise_4_significant_parts("0.0") == 0
# assert exercise_4_significant_parts("3.0.1") == 3

## Bài 5 — Mini challenge: phần rất dài, không được đổi sang `int`

Giả sử mỗi phần có thể dài hàng nghìn chữ số và **không** được dùng `int(...)`. So sánh hai chuỗi số: bỏ số 0 đầu, chuỗi **dài hơn** lớn hơn; dài bằng nhau thì so **từ điển**.

In [20]:
def exercise_5_compare_big(version1: str, version2: str) -> int:
    # TODO
    pass


# assert exercise_5_compare_big("1.2", "1.10") == -1
# assert exercise_5_compare_big("1.0001", "1.1") == 0
# assert exercise_5_compare_big("9" * 50 + ".1", "1" + "0" * 50) == -1
# assert exercise_5_compare_big("1.0.0", "1") == 0

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_compare` sau khi tự làm.

**Bài 3**
- Mức 1: `key` = tuple các số **đã bỏ số 0 cuối**; `sorted` của Python ổn định.
- Mức 2: hoặc `sorted(versions, key=functools.cmp_to_key(compare))`.

**Bài 4**
- Mức 1: đổi sang list số, `pop` các số 0 ở cuối.

**Bài 5**
- Mức 1: hàm `cmp_num(x, y)`: `x = x.lstrip("0")`, `y = y.lstrip("0")`; so `len` trước, rồi so chuỗi.
- Mức 2: phần thiếu coi là `"0"` (sau `lstrip` thành `""`).

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
print("→", trace_compare("7.5.2.4", "7.5.3"))
print("→", trace_compare("1.0.0.1", "1"))

   phần 1: 7 vs 7 → bằng, đi tiếp
   phần 2: 5 vs 5 → bằng, đi tiếp
   phần 3: 2 vs 3 → v1 nhỏ hơn → -1
→ -1
   phần 1: 1 vs 1 → bằng, đi tiếp
   phần 2: 0 vs 0 → bằng, đi tiếp
   phần 3: 0 vs 0 → bằng, đi tiếp
   phần 4: 1 vs 0 → v1 lớn hơn → 1
→ 1


## Đáp án Bài 2

In [22]:
def answer_2_compare_version(version1: str, version2: str) -> int:
    return Solution().compareVersion(version1, version2)

assert answer_2_compare_version("1.2", "1.10") == -1
assert answer_2_compare_version("1.01", "1.001") == 0
assert answer_2_compare_version("1.0", "1.0.0.0") == 0
assert answer_2_compare_version("1.0.1", "1") == 1
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [23]:
def normalized(v: str):
    parts = [int(x) for x in v.split(".")]
    while parts and parts[-1] == 0:
        parts.pop()
    return tuple(parts)

def answer_3_sort_versions(versions: List[str]) -> List[str]:
    return sorted(versions, key=normalized)

assert answer_3_sort_versions(["1.10", "1.2", "1.0.5", "1"]) == ["1", "1.0.5", "1.2", "1.10"]
assert answer_3_sort_versions(["1.0", "1", "0.9"]) == ["0.9", "1.0", "1"]
via_cmp = sorted(["1.10", "1.2", "1.0.5", "1"], key=functools.cmp_to_key(compare_two_pointers))
print("bằng cmp_to_key:", via_cmp)
print("Bài 3 PASS")

bằng cmp_to_key: ['1', '1.0.5', '1.2', '1.10']
Bài 3 PASS


## Đáp án Bài 4

In [24]:
def answer_4_significant_parts(version: str) -> int:
    return len(normalized(version))

assert answer_4_significant_parts("1.2.0.0") == 2
assert answer_4_significant_parts("0.0") == 0
assert answer_4_significant_parts("3.0.1") == 3
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [25]:
def cmp_num(x: str, y: str) -> int:
    x, y = x.lstrip("0"), y.lstrip("0")
    if len(x) != len(y):
        return -1 if len(x) < len(y) else 1
    if x != y:
        return -1 if x < y else 1          # cùng độ dài: so từ điển = so số
    return 0

def answer_5_compare_big(version1: str, version2: str) -> int:
    for x, y in zip_longest(version1.split("."), version2.split("."), fillvalue="0"):
        c = cmp_num(x, y)
        if c:
            return c
    return 0

assert answer_5_compare_big("1.2", "1.10") == -1
assert answer_5_compare_big("1.0001", "1.1") == 0
assert answer_5_compare_big("9" * 50 + ".1", "1" + "0" * 50) == -1
assert answer_5_compare_big("1.0.0", "1") == 0
for _ in range(5000):
    v1, v2 = random_version(), random_version()
    assert answer_5_compare_big(v1, v2) == reference(v1, v2)
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Tách theo `'.'`, đổi từng phần sang **số** (tự bỏ số 0 đầu), phần thiếu coi là `0` (`zip_longest(..., fillvalue=0)`), so từng cặp từ trái sang. Hoặc hai con trỏ đọc số trực tiếp để có `O(1)` bộ nhớ. Không so chuỗi theo từ điển, không so list mà quên số 0 cuối.

---

## Checklist trước khi nộp

- [ ] So phần dưới dạng **số**?
- [ ] Phần thiếu coi là 0 (`"1.0" == "1"`)?
- [ ] Số 0 đầu được bỏ (`"01" == "001"`)?
- [ ] Trả đúng `-1`, `0`, `1`?